In [36]:
import numpy as np
import torch
import ctypes
from time import perf_counter as pc
import polars as pl

In [5]:
def torch_bf16_as_np_bytes(bf16_tensor):
    # If not contiguous this is all pointless as we'll copy
    if not bf16_tensor.is_contiguous():
        bf16_tensor = bf16_tensor.contiguous()
    # Grab ptr + len for byte array for numpy to hold - no copy
    ptr = bf16_tensor.untyped_storage().data_ptr()
    n_bytes = bf16_tensor.numel() * bf16_tensor.element_size()
    array_type = ctypes.c_byte * n_bytes
    buf = array_type.from_address(ptr)
    np_array = np.ctypeslib.as_array(buf)
    return np_array

def np_bytes_as_torch_bf16(uint8_ndarray):
    # From + re-interpret type + re-shape - no copy
    bf16_tensor = (
        torch.from_numpy(uint8_ndarray)
        .view(dtype=torch.bfloat16)
        .view(sh)
    )
    return bf16_tensor

~1GB Video segment kind of tensor shape

In [7]:
sh = (8, 100, 3, 500, 500)

Randomly allocating this tensor ~1s on MBP

In [8]:
%%time
t = torch.rand(sh, dtype=torch.bfloat16)

CPU times: user 1.05 s, sys: 103 ms, total: 1.16 s
Wall time: 1.16 s


Normal roundtrip cast, numpy, back, cast

~200ms normally (500-900ms before as well)

In [18]:
%%timeit
_ = torch.from_numpy(t.to(dtype=torch.float32).numpy()).to(dtype=torch.bfloat16)

150 ms ± 1.37 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [11]:
expected_gb = np.prod(sh) * 2 / 1024 / 1024 / 1024
actual_gb = t.nbytes / 1024 / 1024 / 1024
print(f"{expected_gb=:0.2f} -> {actual_gb=:0.2f}")

expected_gb=1.12 -> actual_gb=1.12


~2ms roundtrip

In [19]:
%%timeit
ndarray_holding_bf16 = torch_bf16_as_np_bytes(t)
and_back_to_bf16_tensor = np_bytes_as_torch_bf16(ndarray_holding_bf16)

1.74 μs ± 23.3 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


Validate roundtrip matches original

In [13]:
%%time
assert torch.allclose(t, and_back_to_bf16_tensor)

CPU times: user 3.97 s, sys: 2.01 s, total: 5.98 s
Wall time: 1.11 s


In [14]:
t[0, 0, 0, 0, :5]

tensor([0.7305, 0.8555, 0.8750, 0.4609, 0.9414], dtype=torch.bfloat16)

In [15]:
and_back_to_bf16_tensor[0, 0, 0, 0, :5]

tensor([0.7305, 0.8555, 0.8750, 0.4609, 0.9414], dtype=torch.bfloat16)

In [ ]:
def zero_copy_roundtrip(t):
    return np_bytes_as_torch_bf16(torch_bf16_as_np_bytes(t))
    
def cast_rountrip(t):
    return torch.from_numpy(t.to(dtype=torch.float32).numpy()).to(dtype=torch.bfloat16) 

In [58]:
n_tests = 10
all_times = []
# for n_bytes in [2 ** i for i in range(10, 31)]:
KB = 1024 ** 1
MB = 1024 ** 2
GB = 1024 ** 3
for n_bytes in [KB, MB, GB]:
    for _ in range(n_tests):
        times = dict()
        times["MB"] = n_bytes / 1024 / 1024
        sh = (n_bytes,)
        
        t0 = pc() 
        t = torch.rand(sh, dtype=torch.bfloat16)
        times["tensor_creation"] = pc() - t0
    
        t0 = pc()
        _ = cast_rountrip(t)
        times["cast_roundtrip"] = pc() - t0
            
        t0 = pc()
        _ = zero_copy_rountrip(t)
        times["zero_copy_roundtrip"] = pc() - t0
        all_times.append(times)

In [59]:
df = pl.DataFrame(all_times)

In [61]:
(
    df
    .with_columns(speedup=pl.col("cast_roundtrip") / pl.col("zero_copy_roundtrip"))
    .group_by("MB")
    .mean()
)

MB,tensor_creation,cast_roundtrip,zero_copy_roundtrip,speedup
f64,f64,f64,f64,f64
1024.0,1.913935,0.266254,0.032005,8.322739
0.000977,0.003606,0.000013,0.000014,1.067689
1.0,0.001906,0.000361,0.000063,6.438815
